# Distributed and federated learning: objectives before architecture

**Core time:** 100 minutes. **Prerequisites:** supervised objectives and gradient descent. **Outcomes:** distinguish centralized, distributed and federated training; define an example-weighted objective; identify statistical/system heterogeneity; separate a local simulation from a deployment.

## 1. Distribution describes more than where code runs

Centralized training places the training data under one logical training process. Data-parallel distributed training commonly divides minibatches across workers and synchronizes gradients under coordinated infrastructure. Federated learning organizes training across clients whose data remain locally held and whose availability, data distributions and resources may differ. Cross-silo settings involve a relatively small set of organizations; cross-device settings may involve many intermittently participating devices. These are useful categories, not strict guarantees about trust or infrastructure.

Federation is not inherently private, secure or fair. Updates can reveal information; some clients can dominate an objective; network participants can be malicious. Keeping raw records local changes the data flow, but the protocol still needs an explicit threat and governance model.

## 2. State whose loss is being minimized

Let client k hold n_k examples and have average local loss F_k. A common global objective weights each local average by its fraction of all examples. An equal-client objective uses weight 1/K instead. The first gives equal weight to examples; the second gives equal weight to clients. Neither is automatically the right policy for every application.

```math
F(w)=\sum_{k=1}^{K}p_kF_k(w),\qquad
F_k(w)=\frac1{n_k}\sum_{i=1}^{n_k}\ell(w;x_{ki},y_{ki}),\qquad
p_k=\frac{n_k}{\sum_j n_j}.
```

Worked example: two clients contain 90 and 10 examples, with local mean losses 0.2 and 1.0. The example-weighted global loss is `0.9*0.2+0.1*1=0.28`; the equal-client mean is 0.6. Reporting only 0.28 hides the smaller client's poor performance. Report both when client equity is relevant and explain which objective trained the model.



In [ ]:
import numpy as np
n = np.array([90,10])
local_loss = np.array([.2,1.])
weighted = np.average(local_loss,weights=n)
macro = local_loss.mean()
assert np.isclose(weighted,.28)
assert np.isclose(macro,.6)
print(weighted,macro)



## 3. Synchronous rounds define a coordination policy

A basic round broadcasts the current model, selects available clients, performs local optimization and aggregates returned updates. The server's round counter is not an epoch counter: clients may take different numbers of steps over different data sizes. Specify whether local work means minibatches, full local epochs or a fixed number of gradient steps.

Synchronous training waits for the chosen participation policy to complete. Stragglers can increase wall-clock duration. Dropping slow clients can bias participation toward particular resources or populations. Asynchronous schemes introduce staleness and a different aggregation process. A single-process simulation can study algorithms and data heterogeneity, but does not measure real network delay, failures or cryptographic overhead.

## 4. Heterogeneity has several axes

Statistical heterogeneity concerns differences in local data distributions, class proportions, sample counts or feature/label relationships. System heterogeneity concerns compute, memory, bandwidth, power and availability. Objective heterogeneity concerns clients needing different models or utility tradeoffs. A benchmark that varies only class proportions does not cover all three.

Before selecting an algorithm, define the intended evaluation population: future observations from current clients, entirely new clients, or a pooled population. These require different splits and metrics. A high pooled score can coexist with poor worst-client performance. New-client generalization cannot be claimed from a test set containing only already participating clients.

## 5. Laboratory scope

The federation lab uses a compact synthetic binary problem and a logistic model with five coefficients including bias. It runs clients sequentially in one CPU process so that aggregation, partitioning and budget accounting are inspectable. It compares a pooled reference, IID FedAvg, non-IID FedAvg and a specified FedProx variant. Communication is calculated from the stated payload format; it is not measured network traffic.

## Exercises and self-check

1. Two clients have 20 and 80 examples and accuracies 0.9 and 0.6. Compute example-weighted and equal-client accuracy. **Check:** 0.66 and 0.75.
2. Explain why ten rounds with five local steps are not equivalent to fifty rounds with one local step. **Check:** aggregation frequency and client drift differ even when a rough local-step count matches.
3. List one statistical, one system and one governance assumption a simulator does not test.
4. Write a deployment question that requires holding out entire clients and another that requires holding out examples within clients.

## Primary reading

- [McMahan et al.: Communication-Efficient Learning of Deep Networks from Decentralized Data](https://proceedings.mlr.press/v54/mcmahan17a.html): identify the iterative model-averaging protocol and its experimental scope.
- [Li et al.: Federated Optimization in Heterogeneous Networks](https://proceedings.mlsys.org/paper_files/paper/2020/hash/1f5fe83998a09396ebe6477d9475ba0c-Abstract.html): separate statistical and system heterogeneity.


# FedAvg: implement and verify a complete round

**Core time:** 2 hours. **Prerequisites:** global/local objectives and gradient descent. **Outcomes:** implement client initialization, local updates and weighted aggregation; distinguish parameters from deltas; test aggregation on an exact numerical case.

## 1. Every selected client starts from the round model

At round t, the server holds w_t. Each selected client receives a copy, performs local optimization on its own training data and returns its updated parameters or an explicitly defined delta. Clients must not accidentally train on a shared mutable array: if client two starts from client one's completed state, the implementation becomes sequential training rather than parallel-from-a-common-start FedAvg.

If clients return parameters, aggregate their returned models with normalized nonnegative weights. If they return deltas `delta_k=w_k-w_t`, average those deltas and add the result to w_t. The two forms are algebraically equivalent when all clients share the same starting point and weights sum to one. Mixing returned parameters and deltas can double-count the server model.

```math
w_{t+1}=\sum_{k\in S_t}\frac{n_k}{\sum_{j\in S_t}n_j}w_{t+1}^{(k)}
=w_t+\sum_{k\in S_t}\frac{n_k}{\sum_{j\in S_t}n_j}\Delta_k.
```

The normalization shown is over participating clients. Under partial/nonuniform participation, this does not automatically yield an unbiased estimate of every desired full-population objective. Sampling and weighting assumptions must be analyzed together.

## 2. Worked example with unequal client sizes

The server starts at `[1,1]`. A client with 3 examples returns `[2,0]`; a client with 1 example returns `[0,4]`. Weighted averaging gives `0.75*[2,0]+0.25*[0,4]=[1.5,1]`. The deltas are `[1,-1]` and `[-1,3]`; their weighted mean is `[0.5,0]`, giving the same final server vector. The equal-client average `[1,2]` corresponds to a different weighting policy.



In [ ]:
import numpy as np
server = np.array([1.,1.])
client_models = np.array([[2.,0.],[0.,4.]])
sizes = np.array([3,1])
weights = sizes/sizes.sum()
parameter_average = weights@client_models
delta_average = server + weights@(client_models-server)
np.testing.assert_allclose(parameter_average,[1.5,1.])
np.testing.assert_allclose(parameter_average,delta_average)
assert np.isclose(weights.sum(),1)
print(parameter_average)



This check should run before training any benchmark. Also test identical client models, a single participating client and rejection of empty client sets or nonpositive total weight.

## 3. One local step connects to centralized gradients

With full participation, a shared initial model, one full-batch local gradient step, common learning rate and example-count weighting, the aggregate update equals one gradient step on the pooled empirical objective. This is a useful invariant for a simulator. It stops holding in that simple form after multiple local steps because each client evaluates later gradients at its own diverging parameters.

For a binary logistic model, each client gradient is `X_k.T@(sigmoid(X_k@w)-y_k)/n_k`. Add a constant-one feature to represent bias. Be explicit about any regularization and whether it applies to bias. The teaching simulator uses no penalty in the basic objective so the weighted-gradient equivalence is easy to verify.

## 4. Local work and optimizer state are algorithm choices

The original family of FedAvg procedures is often described using local epochs/minibatches. Our introductory simulator uses a fixed number of full local gradient steps. This is an intentionally simple variant, not a numerical reproduction of the original deep-network experiments. Documenting that difference is part of correct scholarship.

Momentum or adaptive optimizer state may be reset each round or retained locally; those choices alter the algorithm. Batch-normalization buffers and other non-parameter state also require an aggregation policy in neural models. Averaging every dictionary entry without checking dtype and meaning can corrupt counters or statistics. The logistic lab avoids these additional states so you can focus on the core update.

## 5. Guided implementation

Run `python lab05_federated.py --output outputs/federated`. Inspect the exact aggregation check and one-step equivalence check. Trace one client round by hand: starting model, local gradient, updated model, sample weight and server result. Confirm client index sets are nonempty, disjoint and cover the intended training pool. Keep validation/test data out of local optimization.

## Exercises and self-check

1. Three clients return scalar models 1,3,5 with counts 2,1,1. Compute the server result. **Check:** 2.5.
2. Show algebraically why normalized parameter and delta averaging agree.
3. A simulator initializes each client with `local=server` and updates in place. What can go wrong? **Check:** shared memory can alter the server and subsequent clients; create a copy.
4. Design a one-step equivalence test using independently computed pooled gradients. Explain why testing only identical client data is weak.

## Primary reading

- [McMahan et al.: FedAvg](https://proceedings.mlr.press/v54/mcmahan17a.html): inspect Algorithm 1 and distinguish local epochs, batch size and participation from our simplified full-gradient laboratory.


# Non-IID data: partitioning, client drift and evaluation

**Core time:** 2 hours. **Prerequisites:** FedAvg and probability. **Outcomes:** distinguish label, quantity and feature skew; build a reproducible partition; diagnose local drift; report pooled and client-level metrics without confusing their weights.

## 1. “Non-IID” is a family of experimental conditions

IID means independently and identically distributed under a specified population model. In practice, an IID client partition often means randomly distributing a fixed pooled dataset among clients. A non-IID partition can concentrate labels, vary sample counts, change feature distributions or change the feature–label relationship. One scalar heterogeneity parameter rarely characterizes all these differences.

For label skew, class proportions differ across clients. For quantity skew, client sizes differ. For feature shift, conditional input distributions can differ even for the same label. A synthetic benchmark generated by sorting labels and assigning shards tests a particular label concentration, not every realistic federation.

## 2. Partitioning is part of the experiment

Split the global pool into training, validation and test before allocating training examples to clients. Save the exact client index lists. Verify no overlap between clients unless the protocol explicitly permits shared data, and verify coverage of the intended training pool. Avoid duplicating rare examples to make every client contain every class without documenting the change.

A Dirichlet label-allocation procedure samples class proportions for clients using a concentration parameter alpha. Smaller alpha often yields more concentrated allocations, but finite sample sizes, number of clients and repair/rejection rules all affect the realized partition. Show actual counts and class histograms; “alpha=0.1” alone is insufficient to reconstruct a dataset.

## 3. Worked example: global balance hides local imbalance

Four clients each hold 50 examples. Their positive counts are 45,40,10,5. Globally there are 100 positives and 100 negatives, yet each client sees a skewed class mixture. A local classifier favoring the local majority can perform well locally and poorly on a pooled population. Conversely, a pooled classifier can hide a consistently weak client behind a high overall average.



In [ ]:
import numpy as np
positive = np.array([45,40,10,5])
sizes = np.full(4,50)
rates = positive/sizes
assert positive.sum()/sizes.sum() == .5
print({'client_positive_rates':rates.tolist(),'global_positive_rate':.5})



When clients have unequal sizes, the mean of client accuracies differs from the accuracy over concatenated examples. A worst-client metric can be unstable when the smallest client has very few test examples; show support counts and uncertainty rather than interpreting one extreme estimate as an exact population property.

## 4. Why multiple local steps create drift

Each client descends its own loss. If local objectives prefer different directions or minima, their parameter trajectories separate between aggregations. Averaging the endpoints is not generally the same as following the pooled gradient trajectory. More local steps can reduce communication rounds per amount of computation but increase this divergence. Whether performance improves depends on optimization, data and budget; there is no universal monotonic rule.

Track the norm of each client delta and its distance or angle from the average update. These are diagnostics, not a complete measure of heterogeneity. A large delta could reflect more local work, a larger learning rate or outlier data. Hold these factors fixed before attributing a difference to label skew.

## 5. Evaluation populations must match the claim

The teaching lab compares IID and skewed client partitions over the same global training pool and uses a fixed held-out global validation/test population. Its client-level test slices are a separate documented allocation of held-out examples. This evaluates the stated synthetic setting; it does not establish performance on unseen organizations or devices.

If the claim concerns unseen clients, hold out entire clients during development. If personalization adapts to a client, split that client's data into adaptation/development and final evaluation portions. Never adapt on the same examples later counted as held-out test performance.

## Guided lab and exercises

Inspect the partition JSON and client histograms from `lab05_federated.py`. Then:

1. Create quantity skew while keeping expected class proportions similar. Explain why it is a different intervention from label skew.
2. Compare one versus five local steps using the same partition and explicitly account for total computation and communication.
3. A Dirichlet sampler produces an empty client. Propose a declared rejection/minimum-size policy. **Check:** silently dropping the client changes the population; resampling with a bounded, recorded rule is one defensible alternative.
4. Explain why changing the partition seed and model seed simultaneously complicates attribution. **Check:** data allocation and optimization randomness both vary.

## Primary reading

- [McMahan et al.: FedAvg experiments](https://proceedings.mlr.press/v54/mcmahan17a.html): compare the paper's partition construction with yours.
- [Li et al.: FedProx](https://proceedings.mlsys.org/paper_files/paper/2020/hash/1f5fe83998a09396ebe6477d9475ba0c-Abstract.html): read the treatment of heterogeneous local objectives and work.


# FedProx, local drift and personalization

**Core time:** 120 minutes. **Prerequisites:** FedAvg, gradients and non-IID partitions. **Outcomes:** derive a proximal local gradient; test the zero-penalty equivalence; distinguish a shared global model from personalized models; design adaptation without test leakage.

## 1. A proximal term anchors local optimization

FedProx modifies a client's local objective by penalizing distance from the current round's server model. The reference is fixed during that client's local work. The proximal gradient is proportional to the difference between the current local parameters and the reference, not to the local parameters alone. Confusing this with ordinary L2 regularization changes the method.

```math
\min_w F_k(w)+\frac\mu2\|w-w_t\|_2^2,
\qquad g_k^{prox}(w)=\nabla F_k(w)+\mu(w-w_t).
```

At the beginning of a round, local w equals w_t, so the proximal contribution is zero. Later it pulls the local trajectory toward the reference. Mu controls the strength. A large mu can limit useful adaptation; a smaller drift is not automatically a better predictive model. Select mu on validation under a documented search budget.

## 2. Worked gradient and an invariant

Suppose the local loss gradient is `[0.3,-0.2]`, local parameters are `[1.2,0.8]`, server reference is `[1,1]` and mu=0.5. The proximal contribution is `[0.1,-0.1]`, so the complete gradient is `[0.4,-0.3]`. With mu=0 and otherwise identical implementation, the local update should match the FedAvg local update.



In [ ]:
import numpy as np
local = np.array([1.2,.8])
reference = np.array([1.,1.])
gradient = np.array([.3,-.2])
proximal_gradient = gradient + .5*(local-reference)
np.testing.assert_allclose(proximal_gradient,[.4,-.3])
np.testing.assert_allclose(gradient+0*(local-reference),gradient)
print(proximal_gradient)



The lab implements this objective with fixed full local gradient steps. That is a transparent teaching simplification; the published FedProx framework addresses broader heterogeneous-work conditions. Do not describe the small simulator as a complete reproduction of the paper's systems experiments.

## 3. Personalization changes the output of training

A shared model produces one parameter vector for all clients. Personalization may fine-tune local copies, learn client-specific heads, combine shared and local parameters or optimize a meta-learning objective. These are different methods with different storage, communication and adaptation costs. “Personalized” is not an explanation of which parameters differ or how they were selected.

A straightforward extension initializes each client from the global model and fine-tunes on that client's adaptation data. It needs a separate local validation policy and an untouched local test set. If the client has very little data, local fine-tuning can overfit. Compare global-only, local-only and personalized references when relevant; otherwise an apparent benefit may simply come from giving one method extra training data or work.

## 4. Evaluate both average benefit and who loses

Report per-client before/after metrics with supports, an example-weighted aggregate, an equal-client average and a clearly defined lower-tail or worst-client summary. Count how many clients improve or worsen. A positive mean improvement can conceal substantial deterioration for a subset. If clients have different objectives, explain whether the same metric is meaningful across all of them.

Use the same adaptation budget for compared personalization methods or state the difference. A new-client evaluation requires clients not used to develop the shared initialization or select adaptation hyperparameters. Reusing their final test outcomes to choose the number of local steps is leakage.

## Guided lab and exercises

Run the FedAvg and FedProx configurations in `lab05_federated.py`. Inspect local delta norms, validation curves and payload accounting. The default mu is a fixed teaching configuration, not a tuned optimum.

1. Verify exact FedAvg equivalence when mu=0 with identical partitions and starting weights.
2. Explain why the proximal term has no effect on the first local gradient step from the reference. **Check:** `w-w_t=0` initially.
3. Design a three-way global/local/personalized comparison with distinct adaptation and test data.
4. A larger mu reduces update norms but lowers accuracy. Is the implementation necessarily wrong? **Check:** no; restricting movement can trade off optimization progress and drift.

## Primary reading

- [Li et al.: Federated Optimization in Heterogeneous Networks](https://proceedings.mlsys.org/paper_files/paper/2020/hash/1f5fe83998a09396ebe6477d9475ba0c-Abstract.html): inspect the proximal objective and assumptions; identify differences from the simplified lab.


# Communication, participation and system budgets

**Core time:** 100–120 minutes. **Prerequisites:** federation rounds and resource units. **Outcomes:** calculate a declared payload model; distinguish bytes, rounds and wall time; identify participation bias; compare compression without ignoring metadata.

## 1. A round is not a unit of communication

Two methods with the same round count can send different numbers of models, different model sizes or different precision formats. Count uplink and downlink separately. State whether a broadcast is counted once at a server interface or once per receiving client; both conventions can be useful but are not interchangeable. Include optimizer state, indices, scales or protocol metadata when relevant.

If a model has P scalar parameters stored in b bytes each and K clients receive/send one dense model per round, a simple per-client-sum payload model is `2*K*P*b` bytes per round. This excludes transport headers, retransmissions, cryptography and other protocol state. Label it an analytical payload estimate rather than measured network traffic.

## 2. Worked accounting example

For 1,000,000 parameters, float32 and ten participating clients, downlink is 40,000,000 bytes and uplink another 40,000,000 bytes under per-client accounting. One round totals 80 MB in decimal units, or about 76.29 MiB. One hundred such rounds total 8 GB before overhead. If only uplink is compressed to one byte per parameter, the downlink remains 40 MB unless it is also changed.



In [ ]:
parameters,clients,bytes_per_scalar,rounds = 1_000_000,10,4,100
per_round = 2*parameters*clients*bytes_per_scalar
print({'payload_bytes_per_round':per_round,
       'payload_MB_per_round':per_round/1e6,
       'payload_MiB_per_round':per_round/2**20,
       'total_payload_GB':per_round*rounds/1e9})



A low-rank or sparse representation may transmit fewer values but needs shapes, indices, scales or factors. For an unstructured sparse update, index storage can dominate at modest sparsity. A compression ratio calculated only from retained floating values is incomplete.

## 3. More local work changes several budgets

Increasing local steps may reduce the rounds needed to reach a target validation quality while increasing client computation and drift. Compare quality against both rounds and cumulative bytes, and report local work. If time matters, measure actual wall-clock behavior in an appropriate system. A sequential CPU simulator's runtime is not the latency of parallel clients over a network.

Define time-to-target with a threshold chosen before evaluation and a maximum budget. If a method never reaches the target, report that fact rather than excluding it from the mean. The target should be based on development criteria, not selected after seeing which method wins.

## 4. Availability can bias the participating population

Suppose well-connected clients participate more often and have systematically different data. The training process may optimize their distribution disproportionately. Uniformly sampling from currently available clients is not the same as uniformly sampling from the full client population. Record eligibility, sampling probabilities, dropout handling and any reweighting assumption.

Straggler mitigation is a policy choice: waiting, deadlines, partial aggregation or asynchronous updates each changes the process. Discarding late updates may improve throughput while excluding important client groups. A benchmark that simulates random dropout independent of data does not demonstrate robustness to data-correlated availability.

## 5. Compression can introduce optimization error

Quantizing or sparsifying a client update changes the vector aggregated by the server. The error can be biased or unbiased depending on the compressor. Error-feedback methods retain residual compression error for future updates, introducing state that must be implemented and reset according to the protocol. A compressed method should be compared at matched payload or quality targets, with the same evaluation population.

## Exercises and self-check

1. A five-parameter logistic model communicates float32 weights with five clients for 30 rounds. Compute dense two-way payload. **Check:** `2*5*5*4*30=6,000` bytes under the stated per-client convention, excluding overhead.
2. Why can fewer rounds still consume more total bytes? **Check:** payload per round or participating-client count may increase.
3. Design a participation experiment where availability depends on class mixture. State how it differs from independent random dropout.
4. A sparse update sends 100 float32 values and 100 int32 indices. What is the minimum represented payload before other metadata? **Check:** 800 bytes, not 400.

## Primary reading

- [McMahan et al.: communication-efficient federation](https://proceedings.mlr.press/v54/mcmahan17a.html): identify which communication reduction is measured in the original experiments.
- [Suresh et al.: Distributed Mean Estimation with Limited Communication](https://proceedings.mlr.press/v70/suresh17a.html): study the relationship between communication and aggregation error.


# Federated security, secure aggregation and privacy boundaries

**Core time:** 120 minutes. **Prerequisites:** threat models, DP and FedAvg. **Outcomes:** distinguish transport security, aggregation confidentiality and DP; explain cancellation of pairwise masks conceptually; identify poisoning and participation assumptions; avoid claiming security from locality alone.

## 1. Draw the information flow

Clients hold data, receive a model, compute updates and communicate with a coordinator. Ask what the coordinator, another client, a network observer or a compromised participant can see and change. Transport encryption protects a communication channel under its security assumptions, but the endpoint still receives the plaintext message. Secure aggregation can hide individual contributions while revealing an aggregate. Differential privacy bounds how released output distributions depend on a protected contribution. These mechanisms solve different problems.

Model updates can contain information about local data. A server that never downloads raw examples may still infer sensitive properties from updates or outputs. The statement “data never leave the device” therefore does not by itself establish a statistical privacy guarantee.

## 2. Worked mask-cancellation intuition

Three clients hold scalar updates a,b,c. Give each client pair a random mask with opposite signs at its two endpoints. Client A sends `a+r_ab+r_ac`, B sends `b-r_ab+r_bc`, and C sends `c-r_ac-r_bc`. Adding all three cancels masks, leaving a+b+c. The arithmetic explains the idea, not a complete secure protocol.



In [ ]:
a,b,c = 2.,5.,-1.
r_ab,r_ac,r_bc = 7.,-3.,4.
messages = [a+r_ab+r_ac,b-r_ab+r_bc,c-r_ac-r_bc]
assert sum(messages) == a+b+c
print({'masked_messages_for_public_toy_example':messages,'sum':sum(messages)})



These fixed public masks provide no real confidentiality. A deployable protocol needs appropriate cryptography, authenticated key establishment, dropout recovery, collusion assumptions, finite-field/encoding rules and implementation review. If one participant disappears, simple cancellation may fail; the original secure-aggregation work handles substantially more than the few lines above.

## 3. An aggregate can still reveal information

If the aggregate contains only one client's update, hiding individual messages does not hide that contribution. If an observer knows all but one contribution, the remaining one can be inferred from the sum. Repeated aggregates with changing cohorts can also reveal information depending on the protocol and observer knowledge. Minimum cohort sizes and cryptographic protections should be analyzed in the actual threat model, not treated as universal privacy guarantees.

DP may be combined with aggregation to limit contribution influence, but the clipping, noise, trust and accounting assumptions must be explicit. Record-level and user-level guarantees are different. Who generates noise, who can remove it and what the server observes before noise are central questions.

## 4. Confidentiality does not ensure honest updates

A malicious client may send a manipulated update rather than follow the prescribed training procedure. Robust aggregation, anomaly detection and validation can address some threats under specific assumptions, but no single rule is universally safe. Secure aggregation can limit inspection of individual updates, creating a design tradeoff with some detection methods. Report assumptions about the fraction of compromised clients and their knowledge/control.

Authentication limits who participates, while authorization limits what an authenticated actor can do. Neither proves that an authorized client sends a statistically benign update. A realistic deployment also needs rate limits, logging, model/version integrity and incident handling. Our local simulator does not implement those production controls and must not be represented as a secure deployment.

## 5. Audit checklist as evidence questions

For a proposed system, identify protected information, adversary access, neighboring relation if DP is claimed, minimum cohort assumptions, dropout behavior, update validation, released artifacts and residual risks. For each claimed guarantee, cite the exact mechanism and assumptions. If a property has not been tested or formally established, label it unresolved rather than assigning it a green checkmark.

## Exercises and self-check

1. Remove client C's message from the toy sum. Explain why masks no longer cancel. **Check:** pairwise terms involving C remain.
2. Why does secure aggregation not by itself prevent poisoning? **Check:** hiding a contribution does not ensure that it was computed honestly.
3. Compare server-added noise and client-added noise under a curious-server threat model. Identify what the server sees before any protection.
4. Write a precise correction to “Federated learning guarantees privacy.” **Check:** federation changes data placement; privacy depends on specified mechanisms and assumptions.

## Primary reading

- [Bonawitz et al.: Practical Secure Aggregation for Privacy-Preserving Machine Learning](https://research.google/pubs/practical-secure-aggregation-for-privacy-preserving-machine-learning/): study the protocol's setting and dropout requirements.
- [Dwork and Roth: differential privacy foundations](https://www.cis.upenn.edu/~aaroth/privacybook.html): distinguish statistical privacy from encrypted aggregation.


# Federated-learning laboratory: a controlled heterogeneity benchmark

**Core time:** 3–4 hours. **Prerequisites:** FL 1–6. **Outcomes:** run a transparent simulator; verify aggregation/partitions; compare IID and non-IID conditions under stated budgets; report client-level outcomes and system limitations.

## 1. Freeze the question and the comparison matrix

The core question is: “How does label-skewed client allocation affect a fixed federated optimization procedure on a small synthetic task?” The baseline matrix includes a pooled logistic reference, IID FedAvg, skewed FedAvg and skewed FedProx. These are fixed teaching configurations, not a claim that one algorithm is best. An assessed extension can vary local steps or proximal strength, but should not simultaneously change the dataset, learning rate and evaluation population.

The synthetic generator, split seed, client allocation and model initialization are recorded. A model includes four input coefficients and a bias. Clients take full local gradient steps; call them steps rather than epochs or minibatches. The simulator runs sequentially on one machine and calculates a declared dense float32 payload estimate.

## 2. Read the checks before the score

Run `python lab05_federated.py --output outputs/federated`. Confirm disjoint global splits, nonempty client training sets, no duplicated training indices, complete training coverage, normalized aggregation weights and the one-step equivalence to a pooled gradient. Confirm mu=0 agrees with the nonproximal update. These checks address plausible algorithm errors more directly than a high final accuracy.

Inspect partition counts and positive-class fractions. The realized skew matters more than its label. If a client is empty, the experiment must reject or explicitly handle it before training; dividing by zero or silently excluding it changes the intended population.

## 3. Compare quality against a budget

Plot validation quality versus round and cumulative payload bytes, and record local work. Test metrics are computed after the fixed round budget or a predeclared validation selection rule; do not pick the best test round. If comparing time-to-target, choose the target on development criteria and report methods that fail to reach it.

The pooled reference answers what the same model can do with centralized access to the training pool. It is a useful reference, not necessarily a feasible deployment option. A local-only reference answers a different question and is valuable for a personalization extension. Explain the legal data access and budget for every baseline.

## 4. Worked interpretation of heterogeneous outcomes

Suppose a skewed federation has good pooled accuracy but poor performance on one small client. The pooled result may be dominated by larger clients. Inspect that client's test support, label mix and uncertainty before diagnosing a systematic failure. A small sample can create an extreme estimate; a real disparity can also be hidden by averaging. Report both the counts and the pattern rather than choosing the narrative that favors the algorithm.

If FedProx reduces local delta norms but does not improve test quality, the result may still support the intended anchoring effect while showing that the chosen strength or setting did not improve the prediction objective. Separate mechanism diagnostics from the final utility claim.

## 5. Reproduction package

Include generator parameters, exact indices, client counts, full configuration, validation histories, final predictions, payload formula, unit tests and the environment. Store the actual outputs, not only plots. A peer should be able to recover a reported aggregate by recomputing it from the client-level counts. Save seed-level results before computing a summary.

An extension should repeat the comparison across at least three partition seeds or clearly justify a smaller scope. Keep training randomness separate when attributing differences to allocation. Because the core logistic simulator uses deterministic full-gradient steps, most variation comes from data generation and partitioning; do not invent stochastic optimizer variability that is absent from the implementation.

## Exercises and self-check

1. Recompute every aggregate from per-client numerator/denominator counts and compare with the saved global metric.
2. Match two methods by payload bytes and explain why their local computation may still differ.
3. Extend the protocol to partial participation with a declared sampling rule. Identify the change in the weighting interpretation.
4. Write a limitation statement separating synthetic data, sequential execution, analytical communication and missing privacy/security mechanisms.

## Submission readiness

Submit a 3–5 page report and reproducible package with at least one controlled extension. Show all compared configurations and explain the role of each baseline. A correct conclusion that a method does not improve the chosen setting is acceptable. Claims about real network performance, secure aggregation or DP require evidence beyond this simulator and should not appear as measured results here.

## Primary reading

- [McMahan et al.: FedAvg](https://proceedings.mlr.press/v54/mcmahan17a.html): locate the exact protocol differences from this laboratory.
- [Li et al.: FedProx](https://proceedings.mlsys.org/paper_files/paper/2020/hash/1f5fe83998a09396ebe6477d9475ba0c-Abstract.html): relate proximal reasoning to the measured local-drift diagnostics.
